# Solar Filament Segmentation — Kaggle runnerThis notebook holds **no logic**. It clones the pipeline from GitHub at a pinnedrevision, installs what the Kaggle image is missing, and calls the same CLIentry points used locally. Every code change is made in the repository; the onlything edited here is the revision below.Requirements: *Internet* enabled in the notebook settings (Settings -> Internet),and a GPU accelerator (T4 x2 or P100) for training.

In [ ]:
# --- the only cell you normally edit -----------------------------------------
REPO_URL = "https://github.com/ShreyPatel1311/solar-filament-segmentation.git"
REVISION = "main"          # branch, tag, or full commit SHA - pin a SHA for a final run
CONFIG   = "configs/unet_resnet34.yaml"
OVERRIDES = []             # e.g. ["train.epochs=25", "data.image_size=768"]
# -----------------------------------------------------------------------------

In [ ]:
import os, subprocess, sys, shutil, pathlib

REPO_DIR = pathlib.Path("/kaggle/working/repo")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)          # always start from a clean checkout

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REVISION], check=True)

commit = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                        check=True, capture_output=True, text=True).stdout.strip()
print("running commit", commit)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
os.environ["PYTHONPATH"] = str(REPO_DIR / "src")  # so the ! scripts below import filseg too

In [ ]:
# Kaggle already ships torch, numpy, opencv and pandas; install only the rest so
# the cell stays fast and no preinstalled CUDA build gets replaced.
!pip install -q --no-deps segmentation-models-pytorch==0.3.4 timm==1.0.9 efficientnet-pytorch==0.7.1 pretrainedmodels==0.4.0
!pip install -q albumentations==1.4.15 pycocotools==2.0.8

In [ ]:
import filseg
from filseg.paths import resolve_paths

paths = resolve_paths()
print("filseg", filseg.__version__)
print("data  ", paths.data_root)
print("train ", paths.train_images, len(list(paths.train_images.glob('*.jpeg'))), "images")
print("test  ", paths.test_images, len(list(paths.test_images.glob('*.jpeg'))), "images")

## Train

In [ ]:
args = ["--config", CONFIG] + [a for o in OVERRIDES for a in ("--set", o)]
!python scripts/train.py {" ".join(args)}

## Validate with the leaderboard metric

In [ ]:
CHECKPOINT = "/kaggle/working/checkpoints/unet_r34_best.pt"
!python scripts/evaluate.py --checkpoint {CHECKPOINT}

## Predict the test set and write `submission.csv`

In [ ]:
!python scripts/predict.py --checkpoint {CHECKPOINT} --out /kaggle/working/submission.csv

In [ ]:
import pandas as pd

submission = pd.read_csv("/kaggle/working/submission.csv")
print(submission.shape, "rows |", submission.filament_id.str.rsplit("_", n=1).str[0].nunique(), "images")
submission.head()